In [ ]:
import pandas as pd
import datasets
import tqdm, os
from collections.abc import Iterable
from langchain_core.documents.base import Document

from datetime import datetime
from copy import deepcopy

from dotenv import load_dotenv
load_dotenv()

# dataset moved from the `ErzhuoShao` org to `cssi`
huggingface_path = "cssi/SciSciGPT-SciSciCorpus"
revision = "475c99a8c2afab3c6a7e2e936d8b44c0137437b3"
sciscicorpus = datasets.load_dataset(
	huggingface_path, split="train", revision=revision
).to_pandas()
assert sciscicorpus.shape[0] == 24858, sciscicorpus.shape

In [ ]:
def filter_nan(d):
    d2 = {}
    for k, v in d.items():
        if k in ['section_summary', 'abstract', 'section_text_token_count']:
            continue
        if k in ['section_text']:
            d2["text"] = v[:25000]
            continue
        if k == "section_id":
            d2["section_id"] = int(v)
            continue

        if k == "date":
            if type(v) == str:
                d2["date"] = v
                d2["year"] = int(v.split("-")[0])
            continue

        if k == "author":
            if type(v) == str:
                d2["author"] = v
                authors = v.split(" and ")
                for i in authors:
                    author_name = " ".join(i.split(", ")[::-1])
                    d2["author: {}".format(author_name)] = True
            continue
        
        if k == "embedding":
            d2[k] = v
            continue

        if k in ["urldate", "number"]:
            continue

        if pd.isna(v):
            continue
        
        else:
            d2[k] = v

        if k == "authors":
            d2["authors"] = [' '.join(i.split(', ')[::-1]) for i in v.split(" and ")]
    return d2

documents = [Document(page_content=i['section_summary'], metadata=filter_nan(i)) for i in sciscicorpus.to_dict('records')]

In [ ]:
# The dataset ships its own `embedding` column (text-embedding-3-large, 3072d),
# verified identical to the vectors previously serving in Pinecone
# (cosine 1.0, max delta 1.8e-09) -- so no re-embedding is needed.
assert all(len(i.metadata['embedding']) == 3072 for i in documents[:10])

In [ ]:
import chromadb
from chroma import config

client = chromadb.PersistentClient(path=config.CHROMA_PATH)
try:
    client.delete_collection(config.CORPUS_COLLECTION)
except Exception:
    pass
collection = client.get_or_create_collection(
	name=config.CORPUS_COLLECTION,
	configuration={"hnsw": dict(config.HNSW_CONFIG)},
	embedding_function=None,
)

In [ ]:
# Chroma rejects a single add() larger than 5461 rows, so insert in chunks.
# `text` is popped into the document body to mirror langchain_pinecone,
# which pops the `text` metadata key into page_content.
batch = {"ids": [], "embeddings": [], "documents": [], "metadatas": []}

def flush():
    if batch["ids"]:
        collection.add(**batch)
        for v in batch.values():
            v.clear()

for i in tqdm.trange(len(documents)):
    document = deepcopy(documents[i])
    embedding = document.metadata.pop("embedding")
    text = document.metadata.pop("text", "")
    url = document.metadata['url']

    batch["ids"].append(url + '::' + str(document.metadata['section_id']))
    batch["embeddings"].append([float(x) for x in embedding])
    batch["documents"].append(text)
    batch["metadatas"].append({k: v for k, v in document.metadata.items() if v is not None})

    if len(batch["ids"]) >= 2000:
        flush()
flush()

In [ ]:
assert collection.count() == 24858, collection.count()
collection.count()